# Notebook Version Marker

This is `lt_formal_finetuning_workflow_cpu_v4.ipynb`.

This version is optimized for CPU-only training as much as possible while still using LoRA.
If you do not see this exact text, you opened the wrong notebook.


# Lithuanian Informal-to-Formal Fine-Tuning Workflow

This notebook is self-contained and designed for weak hardware.

What changed in this CPU-friendly version:

- smaller multilingual model,
- LoRA is still used,
- very light default settings,
- optional quick subset mode,
- visible training progress bar,
- no `trl` dependency.


## 1. Install dependencies

In [15]:
%pip install torch transformers datasets peft accelerate sentencepiece safetensors

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and paths

In [16]:
import json
import os
import random
import re
import statistics
from difflib import SequenceMatcher
from pathlib import Path

os.environ['PYTHONUTF8'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'

import torch
from datasets import Dataset
from peft import AutoPeftModelForSeq2SeqLM, LoraConfig, TaskType, get_peft_model
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

project_dir = Path.cwd()
dataset_path = project_dir / 'lt_formal_dataset_303.json'
prepared_dir = project_dir / 'prepared'
outputs_dir = project_dir / 'outputs'
model_output_dir = outputs_dir / 'mt5_small_lora_cpu'
prepared_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

print('Project directory:', project_dir)
print('Dataset exists:', dataset_path.exists())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
else:
    print('CPU-only mode detected.')


Project directory: c:\Users\kipra\OneDrive\Dokumente\New project 6
Dataset exists: True
CUDA available: False
CPU-only mode detected.


## 3. Helper functions

In [ ]:
PROMPT_PREFIX = "Perrašyk šį tekstą taisyklinga ir formalia lietuvių kalba. Ištaisyki gramatiką, skyrybą ir žargoną:"


def load_rows(path: Path) -> list[dict]:
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


def normalize_text(text: str) -> str:
    text = text.casefold().strip()
    return re.sub(r'\s+', ' ', text)


def evaluate_predictions(reference_rows: list[dict], predictions: list[dict]) -> dict:
    prediction_map = {int(row['id']): row['prediction'] for row in predictions}
    scored_rows = []
    exact_match = 0
    normalized_exact_match = 0
    char_similarity_total = 0.0

    for row in reference_rows:
        predicted = prediction_map.get(int(row['id']), '')
        gold = row['formal_text']
        is_exact = predicted == gold
        is_normalized_exact = normalize_text(predicted) == normalize_text(gold)
        similarity = SequenceMatcher(None, normalize_text(predicted), normalize_text(gold)).ratio()

        exact_match += int(is_exact)
        normalized_exact_match += int(is_normalized_exact)
        char_similarity_total += similarity

        scored_rows.append({
            'id': row['id'],
            'informal_text': row['informal_text'],
            'gold': gold,
            'prediction': predicted,
            'exact_match': is_exact,
            'normalized_exact_match': is_normalized_exact,
            'char_similarity': round(similarity, 4),
        })

    total = len(reference_rows)
    return {
        'total_examples': total,
        'exact_match': round(exact_match / total, 4),
        'normalized_exact_match': round(normalized_exact_match / total, 4),
        'average_char_similarity': round(char_similarity_total / total, 4),
        'details': scored_rows,
    }


class NotebookProgressCallback(TrainerCallback):
    def __init__(self):
        self.progress_bar = None
        self.last_epoch_printed = None

    def on_train_begin(self, args, state, control, **kwargs):
        total_steps = state.max_steps if state.max_steps is not None else 0
        print(f'Training started. Total epochs: {args.num_train_epochs}, total optimization steps: {total_steps}')
        self.progress_bar = tqdm(total=total_steps, desc='Training', leave=True)

    def on_step_end(self, args, state, control, **kwargs):
        if self.progress_bar is not None:
            self.progress_bar.n = state.global_step
            epoch_value = 0.0 if state.epoch is None else state.epoch
            self.progress_bar.set_postfix(epoch=f'{epoch_value:.2f}')
            self.progress_bar.refresh()

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch_value = 0.0 if state.epoch is None else state.epoch + 1
        epoch_number = int(epoch_value)
        if self.last_epoch_printed != epoch_number:
            print(f'Starting epoch {epoch_number}/{int(args.num_train_epochs)}')
            self.last_epoch_printed = epoch_number

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            cleaned = {k: round(v, 4) if isinstance(v, float) else v for k, v in logs.items()}
            print(cleaned)

    def on_train_end(self, args, state, control, **kwargs):
        if self.progress_bar is not None:
            self.progress_bar.n = state.global_step
            self.progress_bar.refresh()
            self.progress_bar.close()
        print('Training finished.')


## 4. Load dataset and inspect it

In [18]:
rows = load_rows(dataset_path)
train_rows = [row for row in rows if row['split'] == 'train']
test_rows = [row for row in rows if row['split'] == 'test']

informal_lengths = [len(row['informal_text']) for row in rows]
formal_lengths = [len(row['formal_text']) for row in rows]

stats = {
    'total_rows': len(rows),
    'train_rows': len(train_rows),
    'test_rows': len(test_rows),
    'avg_informal_chars': round(statistics.mean(informal_lengths), 2),
    'avg_formal_chars': round(statistics.mean(formal_lengths), 2),
    'median_informal_chars': statistics.median(informal_lengths),
    'median_formal_chars': statistics.median(formal_lengths),
}
stats

{'total_rows': 303,
 'train_rows': 242,
 'test_rows': 61,
 'avg_informal_chars': 73.56,
 'avg_formal_chars': 76.81,
 'median_informal_chars': 44,
 'median_formal_chars': 54}

In [19]:
rows[:3]

[{'id': 1,
  'informal_text': 'LRT melagiai, Iranas atakavo JAV karinės bazes Emyratuose. Eilinį kartą stengetes suklaidinti Lietuvos piliečius.',
  'formal_text': 'LRT esate melagiai, Iranas atakavo JAV karines bazes Emiratuose. Eilinį kartą stengiates suklaidinti Lietuvos piliečius.',
  'split': 'train'},
 {'id': 2,
  'informal_text': 'Ką konservai kapeikų papylė',
  'formal_text': 'Ką, konservatoriai centų papylė?',
  'split': 'train'},
 {'id': 3,
  'informal_text': 'neispruses melagis',
  'formal_text': 'Neišprūsęs melagis.',
  'split': 'train'}]

## 5. CPU-friendly training configuration

In [29]:
BASE_MODEL = 'google/mt5-small'
NUM_EPOCHS = 10
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
MAX_INPUT_LENGTH = 96
MAX_TARGET_LENGTH = 96
USE_QUICK_CPU_MODE = False # Set to True for faster CPU-based testing, False for full training
MAX_TRAIN_SAMPLES = 96 if USE_QUICK_CPU_MODE else None
MAX_TEST_SAMPLES = 24 if USE_QUICK_CPU_MODE else None

print('Base model:', BASE_MODEL)
print('Quick CPU mode:', USE_QUICK_CPU_MODE)
print('Max train samples:', MAX_TRAIN_SAMPLES)
print('Max test samples:', MAX_TEST_SAMPLES)


Base model: google/mt5-small
Quick CPU mode: False
Max train samples: None
Max test samples: None


## 6. Prepare training subsets

In [30]:
working_train_rows = train_rows[:MAX_TRAIN_SAMPLES] if MAX_TRAIN_SAMPLES is not None else train_rows
working_test_rows = test_rows[:MAX_TEST_SAMPLES] if MAX_TEST_SAMPLES is not None else test_rows

print('Working train rows:', len(working_train_rows))
print('Working test rows:', len(working_test_rows))


Working train rows: 242
Working test rows: 61


## 7. Hardware note

In [31]:
device_type = 'GPU' if torch.cuda.is_available() else 'CPU'
print('Training device:', device_type)
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('This notebook is configured for CPU-friendly training, but it may still take some time.')


Training device: CPU
This notebook is configured for CPU-friendly training, but it may still take some time.


## 8. Fine-tune with LoRA

In [32]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=['q', 'v'],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


def preprocess_example(example: dict) -> dict:
    source_text = f"{PROMPT_PREFIX}\n\n{example['informal_text']}"
    model_inputs = tokenizer(
        source_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding='max_length',
    )
    labels = tokenizer(
        text_target=example['formal_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding='max_length',
    )
    label_ids = labels['input_ids']
    label_ids = [token_id if token_id != tokenizer.pad_token_id else -100 for token_id in label_ids]
    model_inputs['labels'] = label_ids
    return model_inputs


train_dataset = Dataset.from_list(working_train_rows)
eval_dataset = Dataset.from_list(working_test_rows)
train_tokenized = train_dataset.map(preprocess_example, remove_columns=train_dataset.column_names)
eval_tokenized = eval_dataset.map(preprocess_example, remove_columns=eval_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = TrainingArguments(
    output_dir=str(model_output_dir),
    num_train_epochs=NUM_EPOCHS,
    learning_rate=5e-4,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    logging_steps=1,
    save_strategy='no',
    eval_strategy='no',
    per_device_eval_batch_size=1,
    report_to='none',
    remove_unused_columns=False,
    disable_tqdm=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    callbacks=[NotebookProgressCallback()],
)

trainer.train()
trainer.save_model(str(model_output_dir))
tokenizer.save_pretrained(str(model_output_dir))

print('Saved model to:', model_output_dir)


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 18738.95it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 172,032 || all params: 300,348,800 || trainable%: 0.0573


Map: 100%|██████████| 61/61 [00:00<00:00, 1075.95 examples/s]


Training started. Total epochs: 10, total optimization steps: 2420


Training:   0%|          | 0/2420 [00:00<?, ?it/s]

...

{'loss': 5.281, 'grad_norm': 12.4465, 'learning_rate': 0.0, 'epoch': 9.9917}
{'loss': '5.281', 'grad_norm': '12.45', 'learning_rate': '6.198e-07', 'epoch': '9.992'}


Training: 100%|█████████▉| 2419/2420 [58:38<00:01,  1.45s/it, epoch=10.00]

{'loss': 6.4907, 'grad_norm': 13.8479, 'learning_rate': 0.0, 'epoch': 9.9959}
{'loss': '6.491', 'grad_norm': '13.85', 'learning_rate': '4.132e-07', 'epoch': '9.996'}


Training: 100%|██████████| 2420/2420 [58:40<00:00,  1.45s/it, epoch=10.00]


{'loss': 5.7445, 'grad_norm': 6.1421, 'learning_rate': 0.0, 'epoch': 10.0}
{'loss': '5.744', 'grad_norm': '6.142', 'learning_rate': '2.066e-07', 'epoch': '10'}
{'train_runtime': 3520.8081, 'train_samples_per_second': 0.687, 'train_steps_per_second': 0.687, 'total_flos': 240159971082240.0, 'train_loss': 7.4249, 'epoch': 10.0}
{'train_runtime': '3521', 'train_samples_per_second': '0.687', 'train_steps_per_second': '0.687', 'train_loss': '7.425', 'epoch': '10'}
Training finished.
Saved model to: c:\Users\kipra\OneDrive\Dokumente\New project 6\outputs\mt5_small_lora_cpu


## 9. Generate predictions

In [24]:
prediction_tokenizer = AutoTokenizer.from_pretrained(model_output_dir, use_fast=True)
prediction_model = AutoPeftModelForSeq2SeqLM.from_pretrained(model_output_dir)
prediction_model.eval()

test_predictions = []
for row in working_test_rows:
    prompt = f"{PROMPT_PREFIX}\n\n{row['informal_text']}"
    encoded = prediction_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH)
    with torch.no_grad():
        output_ids = prediction_model.generate(
            **encoded,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
        )
    prediction = prediction_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    test_predictions.append({'id': row['id'], 'prediction': prediction})

write_json(outputs_dir / 'test_predictions_cpu_v4.json', test_predictions)
print('Saved predictions:', outputs_dir / 'test_predictions_cpu_v4.json')
len(test_predictions)


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 13949.29it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Saved predictions: c:\Users\kipra\OneDrive\Dokumente\New project 6\outputs\test_predictions_cpu_v4.json


61

## 10. Evaluate predictions

In [25]:
evaluation_report = evaluate_predictions(working_test_rows, test_predictions)
write_json(outputs_dir / 'evaluation_report_cpu_v4.json', evaluation_report)
{k: v for k, v in evaluation_report.items() if k != 'details'}


{'total_examples': 61,
 'exact_match': 0.0,
 'normalized_exact_match': 0.0,
 'average_char_similarity': 0.49}

In [26]:
evaluation_report['details'][:5]

[{'id': 4,
  'informal_text': 'Atleido specialistus ir sukyšo savus kurie geležiinkelį matė tik kompe.',
  'gold': 'Atleido specialistus ir sukišo savus, kurie geležinkelį matę tik kompiuteryje.',
  'prediction': '<extra_id_0> ir sukyšo savus kurie geleži kalba tik kompe.',
  'exact_match': False,
  'normalized_exact_match': False,
  'char_similarity': 0.6471},
 {'id': 13,
  'informal_text': 'Sutapimai?? durpynas dega,dabar miskas,traukiniai nuo begiu..kas toliau?',
  'gold': 'Sutapimai? Durpynas dega, dabar miškas. Traukiniai nuo bėgių nurieda, kas toliau?',
  'prediction': '<extra_id_0>..kas toliau?',
  'exact_match': False,
  'normalized_exact_match': False,
  'char_similarity': 0.3208},
 {'id': 14,
  'informal_text': 'Panašu į aktoriau kauke na',
  'gold': 'Panašu į aktoriaus kaukę.',
  'prediction': '<extra_id_0>. Panašu į aktoriau kauke našu.',
  'exact_match': False,
  'normalized_exact_match': False,
  'char_similarity': 0.6765},
 {'id': 16,
  'informal_text': 'Tai stebės pas k

## 11. Test on a new input

In [27]:
demo_text = 'nu ka cia dabar padarei'

In [28]:
demo_prompt = f"{PROMPT_PREFIX}\n\n{demo_text}"
demo_encoded = prediction_tokenizer(demo_prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH)
with torch.no_grad():
    demo_output_ids = prediction_model.generate(
        **demo_encoded,
        max_new_tokens=MAX_TARGET_LENGTH,
        do_sample=False,
    )
demo_prediction = prediction_tokenizer.decode(demo_output_ids[0], skip_special_tokens=True).strip()
print('Input:', demo_text)
print('Output:', demo_prediction)


Input: nu ka cia dabar padarei
Output: <extra_id_0>. <extra_id_1>. <extra_id_2> dabar padarei. nu ka cia dabar padarei. nu ka cia dabar padarei. nu ka cia dabar padarei. nu ka cia dabar padarei. nu ka cia dabar padarei.
